## 📊 Construction of Subject-Specific Graphs per Connectivity Layer

This script builds graph-based representations of brain networks for each subject in the Naples cohort, separately for each layer (FA, GM, rsfMRI). These graphs are used as input for Graph Neural Networks (GNNs) in classification tasks.

### 🔧 Inputs:
- **Nodal metrics**: Graph-theoretic metrics (degree, strength, betweenness, etc.) per node, per layer and subject.
- **Clinical labels**: Binary classification target (0 = Control, 1 = MS).
- **Significant connections**: For FA and GM, a predefined list of significant connections is used.

### 🧠 Layers Processed:
- **FA (Fractional Anisotropy)** – Structural connectivity.
- **GM (Gray Matter)** – Morphological connectivity.
- **rsfMRI** – Functional connectivity.

### 🧱 Methodology:
1. For each subject and layer:
   - Extract 76 nodes and their 5 nodal metrics.
   - Create an edge list (`edge_index`):
     - For FA and GM: use significant connections from a precomputed list.
     - For rsfMRI: use a fully connected undirected graph.
2. Convert each subject's connectivity profile into a PyTorch Geometric `Data` object:
   - `x`: node features (graph metrics).
   - `edge_index`: connection structure.
   - `y`: subject's label (Control vs MS).
3. Store results in `data_lists[layer]`, allowing separate analysis per modality.

### ✅ Output:
A dictionary `data_lists` with one list of `Data` objects per layer. Each element corresponds to a subject and is ready to be used in a GNN model.

```python
data_lists["FA"]       # List of subject-level graphs for FA
data_lists["GM"]       # List for GM
data_lists["rsfMRI"]   # List for functional networks


## 🧠 GNN Data Preparation – Nodal Features and Graph Construction (Naples Cohort)

This block prepares the input data for training Graph Neural Networks (GNNs) using subject-specific brain graphs for each connectivity layer (FA, GM, rsfMRI). Each subject is represented as a PyTorch Geometric `Data` object.

### 🔧 Key Steps:

1. **Load nodal metrics and clinical labels**:
   - Reads nodal graph metrics per subject from a `.csv` file.
   - Loads clinical metadata and maps subject IDs to class labels (`0` = Control, `1` = MS).

2. **Index decoding function**:
   - Converts a flat vector index (from upper triangle) into matrix coordinates `(i, j)`.

3. **Build graphs per layer**:
   - For FA and GM, edges are defined from significant connections loaded from Excel files.
   - For rsfMRI (no significant edges), a fully connected undirected graph is used.
   - Edges are bidirectional.

4. **Graph creation per subject**:
   - Node features `x` consist of 5 nodal metrics: degree, strength, betweenness, closeness, eigenvector.
   - Graphs are labeled with the clinical group `y`.
   - Each graph includes `edge_index`, `x`, and `y`.

5. **Output**:
   - Stores a list of graphs (`data_list`) per layer in the `data_lists` dictionary.
   - These graphs are ready for GNN training and evaluation.



In [1]:
# Required libraries
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data

# ----------------------------------------
# 1. Define input paths
# ----------------------------------------


nodal_metrics_path = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/nodal_metrics_with_subjects.csv"
clinical_csv_path = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/CLINIC_Naples_B.csv"
conn_paths = {
    "FA": "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/output/FA_significant_connections.xlsx",
    "GM": "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/output//GM_significant_connections.xlsx"
}
clinical_label_col = "GROUP"  # 0 = Control, 1 = EM

# ----------------------------------------
# 2. Load nodal metrics and clinical data
# ----------------------------------------

nodal_df = pd.read_csv(nodal_metrics_path)
clinical_df = pd.read_csv(clinical_csv_path)
clinical_df['ID'] = clinical_df['ID'].astype(str)
id_to_label = dict(zip(clinical_df['ID'], clinical_df[clinical_label_col]))

print(f"✅ Loaded {len(nodal_df['Subject'].unique())} subjects from nodal metrics.")
print(f"✅ Loaded {len(id_to_label)} clinical labels.")

# ----------------------------------------
# 3. Function to convert flat index to (i,j)
# ----------------------------------------

def index_to_ij(index, N):
    i = int(np.floor((2*N - 1 - np.sqrt((2*N - 1)**2 - 8*index)) / 2))
    j = int(index - i*N + i*(i + 1)//2 + i + 1)
    return i, j

# ----------------------------------------
# 4. Loop over layers and build data_list
# ----------------------------------------

N = 76  # number of brain regions
metric_cols = ["degree", "strength", "betweenness", "closeness", "eigenvector"]
layers = ["FA", "GM", "rsfMRI"]
data_lists = {}

for layer in layers:
    print(f"\n🧠 Processing layer: {layer}")

    # Filter nodal metrics for this layer
    layer_df = nodal_df[nodal_df["Layer"] == layer]

    # Load significant connections if available
    if layer in conn_paths:
        sig_conn_df = pd.read_excel(conn_paths[layer])
        sig_conn_indices = [int(c) for c in sig_conn_df.columns[1:]]  # skip index col
        sig_edges = [index_to_ij(idx, N) for idx in sig_conn_indices]
        edge_index = torch.tensor(sig_edges, dtype=torch.long).T
        edge_index = torch.cat([edge_index, edge_index[[1, 0], :]], dim=1)  # bidirectional
        print(f"🔗 Loaded {edge_index.shape[1]//2} significant edges.")
    else:
        # Fully connected for rsfMRI (bidirectional)
        edge_index = torch.combinations(torch.arange(N), r=2).T
        edge_index = torch.cat([edge_index, edge_index[[1, 0], :]], dim=1)
        print(f"🔗 Using fully connected graph ({edge_index.shape[1]//2} undirected edges).")

    # Create graph per subject
    data_list = []
    for subj in layer_df["Subject"].unique():
        subj_data = layer_df[layer_df["Subject"] == subj].sort_values("Node")

        if subj not in id_to_label:
            continue  # skip subjects without label

        x = torch.tensor(subj_data[metric_cols].values, dtype=torch.float32)
        y = torch.tensor([id_to_label[subj]], dtype=torch.long)

        data = Data(x=x, edge_index=edge_index, y=y)
        data.subject_id = subj
        data_list.append(data)

    data_lists[layer] = data_list
    print(f"✅ Created {len(data_list)} graph objects for layer: {layer}")


✅ Loaded 105 subjects from nodal metrics.
✅ Loaded 105 clinical labels.

🧠 Processing layer: FA
🔗 Loaded 697 significant edges.
✅ Created 105 graph objects for layer: FA

🧠 Processing layer: GM
🔗 Loaded 93 significant edges.
✅ Created 105 graph objects for layer: GM

🧠 Processing layer: rsfMRI
🔗 Using fully connected graph (2850 undirected edges).
✅ Created 105 graph objects for layer: rsfMRI


In [2]:
data = data_lists["FA"][0]  # example graph object
print(data)  # print the graph object

print("📌 Node features (x):")
print(data.x[:5])  # print first 5 node features

print("\n🔗 Edge index:")
print(data.edge_index[:, :10])  # print first 10 edges
print("\n🧪 Label:")
print(data.y.item())

print("\n🆔 Subject ID:")
print(data.subject_id)

num_nodes = data.x.shape[0]
num_edges = data.edge_index.shape[1] // 2  # undirected edges

print(f"🔢 Nodes: {num_nodes}, Edges: {num_edges}")

print("🔍 Available attributes:")
print(data.keys)



Data(x=[76, 5], edge_index=[2, 1394], y=[1], subject_id='sub-0001')
📌 Node features (x):
tensor([[4.6000e+01, 2.8077e+01, 0.0000e+00, 7.2115e-01, 1.0125e-01],
        [5.5000e+01, 3.3237e+01, 1.1532e-02, 7.8947e-01, 1.2208e-01],
        [4.9000e+01, 2.8434e+01, 1.5856e-02, 7.4257e-01, 1.0141e-01],
        [1.9000e+01, 1.0559e+01, 0.0000e+00, 5.6818e-01, 3.9147e-02],
        [5.1000e+01, 3.1232e+01, 2.4865e-02, 7.5758e-01, 1.1110e-01]])

🔗 Edge index:
tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 1,  7, 15, 16, 17, 20, 22, 23, 26, 27]])

🧪 Label:
1

🆔 Subject ID:
sub-0001
🔢 Nodes: 76, Edges: 697
🔍 Available attributes:
<bound method BaseData.keys of Data(x=[76, 5], edge_index=[2, 1394], y=[1], subject_id='sub-0001')>


## 🧠 2D Visualization of Nodal Differences – Layer-wise Metric Maps (Naples)

This block generates 2D axial-view plots of group differences in nodal graph metrics (Control − MS) for each connectivity layer and metric, using MNI coordinates of brain regions.

### 🔍 Main Steps:

1. **Load MNI coordinates**:
   - Reads `.node` file with 3D coordinates (if not already loaded).
   - Extracts the X, Y, Z positions of 76 brain regions used for plotting.

2. **Prepare output folder**:
   - Sets up directory to store generated figures per layer and metric.

3. **Compute group-wise differences**:
   - For each layer (`FA`, `GM`, `rsfMRI`) and each metric (`Degree`, `Strength`, etc.):
     - Averages nodal values across subjects for Control and MS groups.
     - Computes the difference $\Delta = \text{Control Mean} - \text{MS Mean}$ per node.

4. **Generate 2D brain plots**:
   - Uses `nilearn.plot_connectome()` with an empty adjacency matrix to show only nodes.
   - Node color reflects $\Delta$:  
     - 🟡 Yellow = higher in Controls  
     - 🟣 Purple = higher in MS  
   - Plots are labeled with the layer and metric name.

5. **Save outputs**:
   - Each figure is saved as `.png` in the specified output directory.
   - Filenames follow the format: `XX_delta_degree_control_minus_ms.png`, etc.



In [3]:
# Required libraries
from nilearn import plotting
import matplotlib.pyplot as plt
import os

# ----------------------------------------
# 1. Load MNI coordinates if not already loaded
# ----------------------------------------

if 'mni_coords' not in globals():
    mni_coords_path = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/Node_mindboggle_default.node"
    with open(mni_coords_path, "r") as f:
        lines = f.readlines()
    mni_coords = []
    for line in lines:
        parts = line.strip().split('\t')
        try:
            coord = list(map(float, parts[:3]))
            mni_coords.append(coord)
        except ValueError:
            continue
    mni_coords = np.array(mni_coords)

# ----------------------------------------
# 2. Define output directory
# ----------------------------------------

output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# 3. Loop over layers and metrics
# ----------------------------------------

metric_names = ["Degree", "Strength", "Betweenness", "Closeness", "Eigenvector"]

for layer in data_lists.keys():
    print(f"\n🧠 Generating plots for layer: {layer}")

    data_list = data_lists[layer]
    control_graphs = [data for data in data_list if data.y.item() == 0]
    ms_graphs = [data for data in data_list if data.y.item() == 1]

    control_features = torch.stack([g.x for g in control_graphs])
    ms_features = torch.stack([g.x for g in ms_graphs])

    for metric_index, metric_name in enumerate(metric_names):
        # Compute Δ = Control − MS
        mean_control = control_features[:, :, metric_index].mean(dim=0).numpy()
        mean_ms = ms_features[:, :, metric_index].mean(dim=0).numpy()
        delta = mean_control - mean_ms

        # Empty adjacency
        empty_adj = np.zeros((76, 76))

        # Plot
        title = f"{layer} - Δ {metric_name} (Control − MS)\nYellow: ↑ Control | Purple: ↑ MS"
        display = plotting.plot_connectome(empty_adj, mni_coords,
                                           node_color=delta,
                                           node_size=40,
                                           edge_threshold=None,
                                           title=title)

        # Save
        filename = f"{layer}_delta_{metric_name.lower().replace(' ', '_')}_control_minus_ms.png"
        output_file = os.path.join(output_dir, filename)
        plt.savefig(output_file, dpi=300)
        plt.close()
        print(f"✅ Saved: {output_file}")



🧠 Generating plots for layer: FA
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_degree_control_minus_ms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_strength_control_minus_ms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_betweenness_control_minus_ms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_closeness_control_minus_ms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_eigenvector_control_minu

This block computes and exports the node-wise group differences in graph metrics between Control and MS subjects for each connectivity layer (FA, GM, rsfMRI).

### 🧠 What it does:

1. **Prepare output directory**:
   - Ensures that the folder for saving CSV files exists.

2. **Loop over connectivity layers**:
   - For each layer, separates graphs into Control and MS groups.
   - Stacks nodal features across subjects to compute group means.

3. **Compute Δ per node and metric**:
   - For each nodal metric (Degree, Strength, etc.):
     - Computes average per node in both groups.
     - Calculates the difference:  
       
       Delta = {Control Mean} - {MS Mean}
       
     - Stores values along with node index, metric name, and layer.

4. **Export results to CSV**:
   - Saves a table of nodal group differences for each layer as a `.csv` file.
   - Filename format: `FA_nodal_deltas_control_minus_ms.csv`, etc.

These CSV files provide structured summaries of how each brain region differs in graph metrics between groups, supporting interpretation, visualization, and downstream analysis.


In [4]:

# Define output directory
output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

# Define metric names
metric_names = ["Degree", "Strength", "Betweenness", "Closeness", "Eigenvector"]

# Loop over each layer
for layer, data_list in data_lists.items():
    print(f"\n📊 Computing nodal deltas for layer: {layer}")
    
    control_graphs = [data for data in data_list if data.y.item() == 0]
    ms_graphs = [data for data in data_list if data.y.item() == 1]

    control_features = torch.stack([g.x for g in control_graphs])
    ms_features = torch.stack([g.x for g in ms_graphs])

    all_deltas = []

    for metric_index, metric_name in enumerate(metric_names):
        mean_control = control_features[:, :, metric_index].mean(dim=0).numpy()
        mean_ms = ms_features[:, :, metric_index].mean(dim=0).numpy()
        delta = mean_control - mean_ms

        for node in range(76):
            all_deltas.append({
                "Layer": layer,
                "Node": node,
                "Metric": metric_name,
                "Mean_Control": mean_control[node],
                "Mean_MS": mean_ms[node],
                "Delta_Control_minus_MS": delta[node]
            })

    # Save to CSV
    delta_df = pd.DataFrame(all_deltas)
    csv_path = os.path.join(output_dir, f"{layer}_nodal_deltas_control_minus_ms.csv")
    delta_df.to_csv(csv_path, index=False)
    print(f"✅ CSV saved: {csv_path}")



📊 Computing nodal deltas for layer: FA
✅ CSV saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_nodal_deltas_control_minus_ms.csv

📊 Computing nodal deltas for layer: GM
✅ CSV saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\GM_nodal_deltas_control_minus_ms.csv

📊 Computing nodal deltas for layer: rsfMRI
✅ CSV saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\rsfMRI_nodal_deltas_control_minus_ms.csv


## 🧠 Visualization of Nodal Differences: RRMS vs SPMS (Naples Cohort)

This block generates 2D brain plots comparing nodal graph metrics between MS subtypes: **RRMS** (Relapsing-Remitting MS) and **SPMS** (Secondary Progressive MS).

### 🧩 Main Objectives:

- Identify brain regions with the greatest differences in nodal metrics between RRMS and SPMS subjects.
- Generate intuitive visualizations for each connectivity layer (FA, GM, rsfMRI) and metric.

### 🔧 Steps Performed:

1. **Define output path**:
   - Sets the folder to save all resulting plots.

2. **Filter subjects by MS subtype**:
   - Extracts subject IDs with labels `RRMS` and `SPMS` from the nodal dataframe.

3. **Loop over each connectivity layer**:
   - For each layer:
     - Splits subjects into RRMS and SPMS.
     - Computes group-wise means per nodal metric (`Degree`, `Strength`, etc.).
     - Calculates Δ = RRMS − SPMS for each node.

4. **Generate 2D plots**:
   - Uses `nilearn.plot_connectome()` with an empty adjacency matrix.
   - Node color encodes Δ values:
     - 🟡 Yellow = Higher in RRMS
     - 🟣 Purple = Higher in SPMS
   - One image per layer and metric.

5. **Save plots**:
   - Saves each visualization as `.png` in the output directory, with descriptive filenames.

✅ These plots reveal subtype-specific differences in brain network topology, supporting deeper clinical stratification within the MS group.


In [4]:
# ----------------------------------------
# 1. Define output directory
# ----------------------------------------

output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# 2. Subjects with RRMS and SPMS labels
# ----------------------------------------

subject_labels = nodal_df[["Subject", "mstype_label"]].drop_duplicates()
rrms_subjects = subject_labels[subject_labels["mstype_label"] == "RRMS"]["Subject"].tolist()
spms_subjects = subject_labels[subject_labels["mstype_label"] == "SPMS"]["Subject"].tolist()

# ----------------------------------------
# 3. Generate RRMS − SPMS plots
# ----------------------------------------

metric_names = ["Degree", "Strength", "Betweenness", "Closeness", "Eigenvector"]

for layer, data_list in data_lists.items():
    print(f"\n🧠 Generating RRMS − SPMS plots for layer: {layer}")

    rrms_graphs = [g for g in data_list if g.subject_id in rrms_subjects]
    spms_graphs = [g for g in data_list if g.subject_id in spms_subjects]

    # Skip if no graphs for one group
    if not rrms_graphs or not spms_graphs:
        print(f"⚠️ Skipping layer {layer} (missing RRMS or SPMS graphs)")
        continue

    rrms_features = torch.stack([g.x for g in rrms_graphs])
    spms_features = torch.stack([g.x for g in spms_graphs])

    for i, metric in enumerate(metric_names):
        mean_rrms = rrms_features[:, :, i].mean(dim=0).numpy()
        mean_spms = spms_features[:, :, i].mean(dim=0).numpy()
        delta = mean_rrms - mean_spms  # RRMS − SPMS

        title = f"{layer} - Δ {metric} per Node (RRMS − SPMS)\nYellow = ↑ RRMS | Purple = ↑ SPMS"
        empty_adj = np.zeros((76, 76))

        display = plotting.plot_connectome(empty_adj, mni_coords,
                                           node_color=delta,
                                           node_size=40,
                                           edge_threshold=None,
                                           title=title)

        filename = f"{layer}_delta_{metric.lower()}_rrms_minus_spms.png"
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, dpi=300)
        plt.close()
        print(f"✅ Saved: {filepath}")



🧠 Generating RRMS − SPMS plots for layer: FA
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_degree_rrms_minus_spms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_strength_rrms_minus_spms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_betweenness_rrms_minus_spms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_closeness_rrms_minus_spms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_eigenvector_rrms

## 🧠 Visualization of Nodal Differences: EDSS Mild vs Severe (Naples Cohort)

This block generates 2D brain plots showing nodal metric differences between MS patients with **mild** (EDSS 0–2) and **severe** (EDSS 6.5–9) levels of disability, based on graph features computed per subject and layer.

### 🎯 Objective:
- Visualize regional differences in nodal graph metrics related to clinical disability severity (EDSS).
- Identify brain regions whose topological role changes with disease progression.

### 🧩 Steps Performed:

1. **Define output path**:
   - Creates the directory to save EDSS comparison plots.

2. **Filter subjects by EDSS group**:
   - Retrieves subject IDs for the groups:
     - **Mild**: EDSS 0–2
     - **Severe**: EDSS 6.5–9

3. **Loop over layers and metrics**:
   - For each layer (`FA`, `GM`, `rsfMRI`) and each nodal metric (`Degree`, `Strength`, etc.):
     - Averages nodal values across mild and severe groups.
     - Computes difference per node:
       \[
       \Delta = \text{Mild Mean} - \text{Severe Mean}
       \]

4. **Generate plots**:
   - Visualizes nodes in 2D (axial view) using `nilearn.plot_connectome()`.
   - No edges are displayed — only nodes.
   - Color encoding:
     - 🟡 Yellow = Higher in Mild
     - 🟣 Purple = Higher in Severe

5. **Save results**:
   - Each figure is saved as `.png` with clear naming by layer and metric.

✅ These plots highlight nodal-level network alterations that correlate with clinical severity, potentially revealing neuroanatomical biomarkers of MS progression.


In [5]:
# ----------------------------------------
# 1. Output directory
# ----------------------------------------

output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# 2. Filter subjects with EDSS labels
# ----------------------------------------

subject_labels = nodal_df[["Subject", "edss_group"]].drop_duplicates()
mild_subjects = subject_labels[subject_labels["edss_group"] == "0–2 (Mild)"]["Subject"].tolist()
severe_subjects = subject_labels[subject_labels["edss_group"] == "6.5–9 (Severe)"]["Subject"].tolist()

# ----------------------------------------
# 3. Metric names
# ----------------------------------------

metric_names = ["Degree", "Strength", "Betweenness", "Closeness", "Eigenvector"]

# ----------------------------------------
# 4. Loop over layers and generate plots
# ----------------------------------------

for layer, data_list in data_lists.items():
    print(f"\n🧠 Generating EDSS Mild − Severe plots for layer: {layer}")

    mild_graphs = [g for g in data_list if g.subject_id in mild_subjects]
    severe_graphs = [g for g in data_list if g.subject_id in severe_subjects]

    # Skip if no graphs for one group
    if not mild_graphs or not severe_graphs:
        print(f"⚠️ Skipping layer {layer} (missing Mild or Severe subjects)")
        continue

    mild_features = torch.stack([g.x for g in mild_graphs])
    severe_features = torch.stack([g.x for g in severe_graphs])

    for i, metric in enumerate(metric_names):
        mean_mild = mild_features[:, :, i].mean(dim=0).numpy()
        mean_severe = severe_features[:, :, i].mean(dim=0).numpy()
        delta = mean_mild - mean_severe

        title = f"{layer} - Δ {metric} per Node (EDSS Mild − Severe)\nYellow = ↑ Mild | Purple = ↑ Severe"
        empty_adj = np.zeros((76, 76))

        display = plotting.plot_connectome(empty_adj, mni_coords,
                                           node_color=delta,
                                           node_size=40,
                                           edge_threshold=None,
                                           title=title)

        filename = f"{layer}_delta_{metric.lower()}_edss_mild_minus_severe.png"
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, dpi=300)
        plt.close()
        print(f"✅ Saved: {filepath}")



🧠 Generating EDSS Mild − Severe plots for layer: FA
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_degree_edss_mild_minus_severe.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_strength_edss_mild_minus_severe.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_betweenness_edss_mild_minus_severe.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_delta_closeness_edss_mild_minus_severe.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/f

## 🧠 GNN Classification with Cross-Validation – Naples Cohort

This block trains and evaluates three Graph Neural Network (GNN) architectures using nodal graph metrics and connectivity graphs per subject. It applies stratified k-fold cross-validation for robust performance estimation across each connectivity layer (FA, GM, rsfMRI).

### 🔧 Key Components:

1. **Model Architectures**:
   - `GCN`: Graph Convolutional Network
   - `GraphSAGE`: Neighborhood aggregation-based model
   - `GAT`: Graph Attention Network with multi-head attention
   - All models include two graph layers and a fully connected layer for binary classification (Control vs MS).

2. **Training and Evaluation Loops**:
   - `train()`: Standard training loop with dropout and Adam optimizer.
   - `test()`: Computes predictions and probabilities on test data; collects metrics.

3. **Cross-Validation Pipeline**:
   - Uses **5-fold stratified cross-validation** to maintain class balance across folds.
   - For each fold:
     - Trains the model on the training subset.
     - Evaluates performance on the test set.
     - Computes and stores metrics: Accuracy, F1, Precision, Recall, AUC.

4. **Results Handling**:
   - Metrics from all folds are saved to a CSV file per model and layer.
   - An average row is added for global comparison.
   - Output files follow the format: `XX_GNN_metrics_folds.csv`, etc.

5. **Device Support**:
   - Automatically uses GPU if available (`cuda`), otherwise falls back to CPU.



In [7]:
# 📦 Required libraries
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
import random


# ----------------------------------------
# 1. Set random seeds
# ----------------------------------------

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# ----------------------------------------
# 2. Define GNN architectures
# ----------------------------------------

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, batch)
        return self.lin(x)

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, batch)
        return self.lin(x)

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=4, concat=True)
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels, heads=1)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, batch)
        return self.lin(x)

# ----------------------------------------
# 3. Train/test loop
# ----------------------------------------

def train(model, loader, optimizer, device):
    model.train()
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        loss = F.cross_entropy(out, data.y)
        loss.backward()
        optimizer.step()

def test(model, loader, device):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            prob = F.softmax(out, dim=1)
            pred = prob.argmax(dim=1)
            y_true.extend(data.y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())
            y_prob.extend(prob[:, 1].cpu().numpy())
    return y_true, y_pred, y_prob

# ----------------------------------------
# 4. Cross-validation by model and layer
# ----------------------------------------

def run_cross_validation(ModelClass, model_name, data_list, layer, output_dir, k=5, hidden_dim=32, epochs=100):
    print(f"\n🔁 Running {model_name} for layer {layer} with {k}-Fold Stratified CV")
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    labels = [data.y.item() for data in data_list]

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(data_list, labels)):
        train_data = [data_list[i] for i in train_idx]
        test_data = [data_list[i] for i in test_idx]

        train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=16)

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = ModelClass(in_channels=5, hidden_channels=hidden_dim).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

        for epoch in range(epochs):
            train(model, train_loader, optimizer, device)

        y_true, y_pred, y_prob = test(model, test_loader, device)

        # Plot confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Control", "MS"])
        disp.plot(cmap="Blues")
        plt.title(f"{model_name} - {layer} - Fold {fold+1}")
        cm_path = os.path.join(output_dir, f"{layer}_{model_name}_fold{fold+1}_confmat.png")
        plt.savefig(cm_path, dpi=150)
        plt.close()
        print(f"✅ Saved confusion matrix: {cm_path}")

        metrics = {
            "Model": model_name,
            "Layer": layer,
            "Fold": fold + 1,
            "Accuracy": accuracy_score(y_true, y_pred),
            "F1": f1_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred),
            "Recall": recall_score(y_true, y_pred),
            "AUC": roc_auc_score(y_true, y_prob)
        }
        fold_results.append(metrics)
        print(f"📊 Fold {fold+1} - Acc: {metrics['Accuracy']:.3f}, F1: {metrics['F1']:.3f}, AUC: {metrics['AUC']:.3f}")

    # Save to CSVs
    df = pd.DataFrame(fold_results)
    avg = df.iloc[:, 3:].mean().to_dict()
    avg_row = {
        "Model": model_name,
        "Layer": layer,
        "Fold": "Average",
        **avg
    }
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)

    print(f"\n📋 Results for {model_name} - {layer}:")
    print(df.round(3).to_string(index=False))

    # Save
    folds_path = os.path.join(output_dir, f"{layer}_GNN_metrics_folds.csv")
    #df.to_csv(folds_path, index=False)
    #print(f"✅ Saved to {folds_path}")
    # Append to CSV if exists, else create with header
    df.to_csv(folds_path, index=False, mode='a', header=not os.path.exists(folds_path))
    print(f"✅ Appended results to {folds_path}")

# ----------------------------------------
# 5. Run for all models and layers
# ----------------------------------------

output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

for layer, data_list in data_lists.items():
    run_cross_validation(GCN, "GCN", data_list, layer, output_dir)
    run_cross_validation(GraphSAGE, "GraphSAGE", data_list, layer, output_dir)
    run_cross_validation(GAT, "GAT", data_list, layer, output_dir)



🔁 Running GCN for layer FA with 5-Fold Stratified CV
✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_GCN_fold1_confmat.png
📊 Fold 1 - Acc: 0.667, F1: 0.533, AUC: 0.682
✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_GCN_fold2_confmat.png
📊 Fold 2 - Acc: 0.762, F1: 0.706, AUC: 0.900
✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_GCN_fold3_confmat.png
📊 Fold 3 - Acc: 0.762, F1: 0.737, AUC: 0.718
✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_GCN_fold4_confmat.png
📊 Fold 4 - Acc: 0.857, 

f:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\rsfMRI_GraphSAGE_fold3_confmat.png
📊 Fold 3 - Acc: 0.381, F1: 0.480, AUC: 0.473
✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\rsfMRI_GraphSAGE_fold4_confmat.png
📊 Fold 4 - Acc: 0.571, F1: 0.609, AUC: 0.636
✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\rsfMRI_GraphSAGE_fold5_confmat.png
📊 Fold 5 - Acc: 0.619, F1: 0.556, AUC: 0.755

📋 Results for GraphSAGE - rsfMRI:
    Model  Layer    Fold  Accuracy    F1  Precision  Recall   AUC
GraphSAGE rsfMRI       1     0.476 0.421      0.500   0.364 0.591
GraphSAGE rsfMRI       2     0.476 0.000      0.000   0.000 0.682
GraphSAGE r

f:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\rsfMRI_GAT_fold2_confmat.png
📊 Fold 2 - Acc: 0.524, F1: 0.375, AUC: 0.591
✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\rsfMRI_GAT_fold3_confmat.png
📊 Fold 3 - Acc: 0.524, F1: 0.000, AUC: 0.664


f:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\rsfMRI_GAT_fold4_confmat.png
📊 Fold 4 - Acc: 0.524, F1: 0.000, AUC: 0.573


f:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✅ Saved confusion matrix: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\rsfMRI_GAT_fold5_confmat.png
📊 Fold 5 - Acc: 0.476, F1: 0.645, AUC: 0.827

📋 Results for GAT - rsfMRI:
Model  Layer    Fold  Accuracy    F1  Precision  Recall   AUC
  GAT rsfMRI       1     0.476 0.000      0.000   0.000 0.582
  GAT rsfMRI       2     0.524 0.375      0.600   0.273 0.591
  GAT rsfMRI       3     0.524 0.000      0.000   0.000 0.664
  GAT rsfMRI       4     0.524 0.000      0.000   0.000 0.573
  GAT rsfMRI       5     0.476 0.645      0.476   1.000 0.827
  GAT rsfMRI Average     0.505 0.204      0.215   0.255 0.647
✅ Appended results to F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\rsfMRI_GNN_metrics_folds.csv


In [8]:
# ----------------------------------------
# 🔍 GNNExplainer on real graph (FA layer, GraphSAGE)
# ----------------------------------------

from torch_geometric.explain import Explainer, GNNExplainer
from torch_geometric.explain.config import ModelConfig
import pandas as pd

# ✅ Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Select layer and a subject graph
target_layer = "FA"
target_data_list = data_lists[target_layer]
target_data = target_data_list[0]  # you can choose any index
target_data = target_data.to(device)

# ✅ Initialize and train model (same as above)
model = GraphSAGE(in_channels=target_data.x.shape[1], hidden_channels=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# Wrap in loader for consistency
loader = DataLoader([target_data], batch_size=1, shuffle=True)
for epoch in range(100):
    train(model, loader, optimizer, device)

model.eval()

# ✅ Configure Explainer
explainer = Explainer(
    model=model,
    algorithm=GNNExplainer(epochs=100),
    explanation_type="model",
    node_mask_type="attributes",
    edge_mask_type="object",
    model_config=ModelConfig(
        mode="binary_classification",
        task_level="graph",
        return_type="raw"
    )
)

# ✅ Run explanation on subject
explanation = explainer(
    x=target_data.x,
    edge_index=target_data.edge_index,
    batch=target_data.batch
)

# ✅ Extract and show results
print(f"🎯 Explained subject: {target_data.subject_id}")
print("🔎 Node feature importance (avg across nodes):")
mean_node_importance = explanation.node_mask.mean(dim=0)
print(mean_node_importance)

print("🧩 Edge importance (top 10):")
top_edges = torch.topk(explanation.edge_mask, 10)
for i in range(10):
    edge_idx = top_edges.indices[i].item()
    score = top_edges.values[i].item()
    edge = (target_data.edge_index[0, edge_idx].item(), target_data.edge_index[1, edge_idx].item())
    print(f"Edge {edge} - importance: {score:.4f}")

# ✅ Save node feature importance
importance_df = pd.DataFrame({
    "Feature": ["degree", "strength", "betweenness", "closeness", "eigenvector"],
    "Importance": mean_node_importance.cpu().numpy()
})
csv_path = os.path.join(output_dir, f"{target_layer}_GraphSAGE_explainer_node_importance.csv")
importance_df.to_csv(csv_path, index=False)
print(f"✅ Node feature importance saved to: {csv_path}")


🎯 Explained subject: sub-0001
🔎 Node feature importance (avg across nodes):
tensor([0.3136, 0.2882, 0.1893, 0.2741, 0.2725])
🧩 Edge importance (top 10):
Edge (68, 62) - importance: 0.3733
Edge (18, 72) - importance: 0.3702
Edge (71, 4) - importance: 0.3683
Edge (22, 42) - importance: 0.3622
Edge (42, 8) - importance: 0.3586
Edge (39, 14) - importance: 0.3570
Edge (14, 28) - importance: 0.3533
Edge (41, 19) - importance: 0.3523
Edge (18, 11) - importance: 0.3503
Edge (25, 41) - importance: 0.3484
✅ Node feature importance saved to: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML\FA_GraphSAGE_explainer_node_importance.csv


In [15]:
# ----------------------------------------
# 🔁 Generalized GNNExplainer for any model and all layers
# ----------------------------------------

def run_explainer_all_layers(model_class, model_name, data_lists, output_dir, epochs=100, hidden_channels=32):
    """
    Train a GNN model per layer and apply GNNExplainer to compute node and edge importance by group.
    
    Args:
        model_class: GNN model class (e.g., GCN, GraphSAGE, GAT)
        model_name: string name for output files
        data_lists: dict of {layer: list of Data objects}
        output_dir: where to save the CSVs
        epochs: number of training epochs
        hidden_channels: hidden dimension for the model
    """
    from torch_geometric.explain import Explainer, GNNExplainer
    from torch_geometric.explain.config import ModelConfig
    from collections import defaultdict
    from tqdm import tqdm

    def collect_node_edge_importance(data_list, model, device="cpu"):
        model.eval()
        explainer = Explainer(
            model=model,
            algorithm=GNNExplainer(epochs=100),
            explanation_type="model",
            node_mask_type="attributes",
            edge_mask_type="object",
            model_config=ModelConfig(
                mode="binary_classification",
                task_level="graph",
                return_type="raw"
            )
        )
        node_results = defaultdict(list)
        edge_results = defaultdict(list)

        for data in tqdm(data_list, desc="🔍 Explaining subjects"):
            label = data.y.item()
            group = "control" if label == 0 else "ms"
            data = data.to(device)
            explanation = explainer(
                x=data.x,
                edge_index=data.edge_index,
                batch=data.batch
            )
            node_results[group].append(explanation.node_mask.detach().cpu())
            edge_results[group].append(explanation.edge_mask.detach().cpu())

        return node_results, edge_results

    def compute_groupwise_importance(node_results, edge_results, edge_index_ref):
        mean_node_control = torch.stack(node_results["control"]).mean(dim=0).mean(dim=0)
        mean_node_ms = torch.stack(node_results["ms"]).mean(dim=0).mean(dim=0)
        delta_node = mean_node_control - mean_node_ms

        df_nodes = pd.DataFrame({
            "feature": ["degree", "strength", "betweenness", "closeness", "eigenvector"],
            "control": mean_node_control.numpy().flatten(),
            "ms": mean_node_ms.numpy().flatten(),
            "delta": delta_node.numpy().flatten()
        })


        mean_edge_control = torch.stack(edge_results["control"]).mean(dim=0)
        mean_edge_ms = torch.stack(edge_results["ms"]).mean(dim=0)
        delta_edge = mean_edge_control - mean_edge_ms

        i = edge_index_ref[0].cpu().numpy()
        j = edge_index_ref[1].cpu().numpy()

        df_edges = pd.DataFrame({
            "edge_idx": np.arange(len(i)),
            "source": i,
            "target": j,
            "control": mean_edge_control.numpy(),
            "ms": mean_edge_ms.numpy(),
            "delta": delta_edge.numpy()
        }).sort_values("delta", ascending=False)

        return df_nodes, df_edges

    # Main loop
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    explainer_out = os.path.join(output_dir, f"GNNExplainer_{model_name}")
    os.makedirs(explainer_out, exist_ok=True)

    print(f"\n🧠 Running GNNExplainer for model: {model_name}")
    for layer, data_list in data_lists.items():
        print(f"\n📍 Layer: {layer} | Subjects: {len(data_list)}")

        # Train model on full data for this layer
        model = model_class(in_channels=5, hidden_channels=hidden_channels).to(device)
        loader = DataLoader(data_list, batch_size=16, shuffle=True)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

        for epoch in range(epochs):
            train(model, loader, optimizer, device)

        print(f"✅ Trained {model_name} on {layer}. Now explaining...")

        node_results, edge_results = collect_node_edge_importance(data_list, model, device=device)
        df_nodes, df_edges = compute_groupwise_importance(node_results, edge_results, data_list[0].edge_index)

        # Save
        df_nodes.to_csv(os.path.join(explainer_out, f"node_importance_{layer}_{model_name}_by_group.csv"), index=False)
        df_edges.to_csv(os.path.join(explainer_out, f"edge_importance_{layer}_{model_name}_by_group.csv"), index=False)

        print(f"💾 Saved results for {layer} → {explainer_out}")

# Define output directory
# "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/graph_metrics/gnn_explainer"
explainer_output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer"
# Run for all models
run_explainer_all_layers(GCN, "GCN", data_lists, explainer_output_dir)
run_explainer_all_layers(GraphSAGE, "GraphSAGE", data_lists, explainer_output_dir)
run_explainer_all_layers(GAT, "GAT", data_lists, explainer_output_dir)




🧠 Running GNNExplainer for model: GCN

📍 Layer: FA | Subjects: 105
✅ Trained GCN on FA. Now explaining...


🔍 Explaining subjects: 100%|██████████| 105/105 [00:51<00:00,  2.02it/s]


💾 Saved results for FA → F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\GNNExplainer_GCN

📍 Layer: GM | Subjects: 105
✅ Trained GCN on GM. Now explaining...


🔍 Explaining subjects: 100%|██████████| 105/105 [00:57<00:00,  1.82it/s]


💾 Saved results for GM → F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\GNNExplainer_GCN

📍 Layer: rsfMRI | Subjects: 105
✅ Trained GCN on rsfMRI. Now explaining...


...

💾 Saved results for FA → F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\GNNExplainer_GAT

📍 Layer: GM | Subjects: 105
✅ Trained GAT on GM. Now explaining...


🔍 Explaining subjects: 100%|██████████| 105/105 [01:27<00:00,  1.21it/s]


💾 Saved results for GM → F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\GNNExplainer_GAT

📍 Layer: rsfMRI | Subjects: 105
✅ Trained GAT on rsfMRI. Now explaining...


🔍 Explaining subjects: 100%|██████████| 105/105 [03:02<00:00,  1.74s/it]

💾 Saved results for rsfMRI → F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\GNNExplainer_GAT


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# === Base route ===
base_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer"  # 🔁 CAMBIA AQUÍ
layers = ["FA", "GM", "rsfMRI"]

# === Output folder ===
plots_dir = os.path.join(base_dir, "plots")
os.makedirs(plots_dir, exist_ok=True)

# === Model folder ===
model_folders = [f for f in os.listdir(base_dir) if f.startswith("GNNExplainer_")]
print("📂 Model folders found:", model_folders)

# === Loop model and layer ===
for model_folder in model_folders:
    model_name = model_folder.replace("GNNExplainer_", "")
    model_path = os.path.join(base_dir, model_folder)

    for layer in layers:
        # === Paths de entrada ===
        node_csv = os.path.join(model_path, f"node_importance_{layer}_{model_name}_by_group.csv")
        edge_csv = os.path.join(model_path, f"edge_importance_{layer}_{model_name}_by_group.csv")

        # === Plot nodal features ===
        if os.path.exists(node_csv):
            node_df = pd.read_csv(node_csv)
            features = node_df["feature"]
            control = node_df["control"]
            ms = node_df["ms"]

            fig, ax = plt.subplots(figsize=(8, 5))
            bar_width = 0.35
            index = range(len(features))
            ax.bar(index, control, bar_width, label="Control", alpha=0.8)
            ax.bar([i + bar_width for i in index], ms, bar_width, label="MS", alpha=0.8)
            ax.set_xlabel("Feature")
            ax.set_ylabel("Importance")
            ax.set_title(f"Nodal Importance: {layer} – {model_name}")
            ax.set_xticks([i + bar_width / 2 for i in index])
            ax.set_xticklabels(features)
            ax.legend()
            ax.grid(axis='y', linestyle='--', alpha=0.5)
            plt.tight_layout()
            fig_path = os.path.join(plots_dir, f"nodal_importance_{layer}_{model_name}.png")
            plt.savefig(fig_path, dpi=150)
            plt.close()
            print(f"✅ Saved: {fig_path}")

        # === Plot top 10 edge deltas ===
        if os.path.exists(edge_csv):
            edge_df = pd.read_csv(edge_csv)
            top_edges = edge_df.sort_values("delta", ascending=False).head(10)

            plt.figure(figsize=(10, 5))
            plt.barh(
                y=[f"{int(row.source)}-{int(row.target)}" for _, row in top_edges.iterrows()],
                width=top_edges["delta"],
                color="darkorange"
            )
            plt.xlabel("Δ Importance (Control − MS)")
            plt.title(f"Top 10 Differential Connections: {layer} – {model_name}")
            plt.gca().invert_yaxis()
            plt.grid(axis="x", linestyle="--", alpha=0.5)
            plt.tight_layout()
            fig_path = os.path.join(plots_dir, f"edge_top10_{layer}_{model_name}.png")
            plt.savefig(fig_path, dpi=150)
            plt.close()
            print(f"✅ Saved: {fig_path}")


📂 Model folders found: ['GNNExplainer_GAT', 'GNNExplainer_GCN', 'GNNExplainer_GraphSAGE']
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\plots\nodal_importance_FA_GAT.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\plots\edge_top10_FA_GAT.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\plots\nodal_importance_GM_GAT.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\plots\edge_top10_GM_GAT.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/

In [19]:

import os
import pandas as pd
import numpy as np
from nilearn import plotting
import matplotlib.pyplot as plt

# === Configuración ===
base_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer" 
node_path = r"F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/Node_mindboggle_default.node"
layers = ["FA", "GM", "rsfMRI"]
output_dir = os.path.join(base_dir, "plots_nilearn_top5+5")
os.makedirs(output_dir, exist_ok=True)

# === Cargar coordenadas ===
coords_df = pd.read_csv(node_path, sep="\t", header=None)
coords_df.columns = ["x", "y", "z", "name", "label", "color"]
coords = coords_df[["x", "y", "z"]].values

# === Recorrer todos los modelos y capas ===
model_folders = [f for f in os.listdir(base_dir) if f.startswith("GNNExplainer_")]

for model_folder in model_folders:
    model_name = model_folder.replace("GNNExplainer_", "")
    model_path = os.path.join(base_dir, model_folder)

    for layer in layers:
        edge_csv = os.path.join(model_path, f"edge_importance_{layer}_{model_name}_by_group.csv")
        if os.path.exists(edge_csv):
            try:
                edge_df = pd.read_csv(edge_csv)

                # === Seleccionar top-5 positivas y top-5 negativas ===
                top_pos = edge_df[edge_df["delta"] > 0].sort_values("delta", ascending=False).head(5)
                top_neg = edge_df[edge_df["delta"] < 0].sort_values("delta", ascending=True).head(5)
                combined = pd.concat([top_pos, top_neg])

                # === Matriz de conectividad ===
                matrix = np.zeros((76, 76))
                for _, row in combined.iterrows():
                    i, j = int(row["source"]), int(row["target"])
                    matrix[i, j] = row["delta"]
                    matrix[j, i] = row["delta"]

                vmax = np.max(np.abs(matrix))

                # === Título informativo ===
                title = f"{layer} – Top-5 ↑ Control & ↑ MS connections\nModel: {model_name}\nRed = ↑ Control | Blue = ↑ MS"

                # === Conectograma
                fig = plotting.plot_connectome(
                    adjacency_matrix=matrix,
                    node_coords=coords,
                    edge_threshold=None,
                    display_mode="lzry",
                    title=title,
                    edge_cmap="bwr",
                    edge_vmin=-vmax,
                    edge_vmax=+vmax,
                    node_color="black",
                    node_size=20,
                    annotate=False
                )

                # === Guardar
                fig_path = os.path.join(output_dir, f"connectome_top5_control_ms_{layer}_{model_name}.png")
                plt.savefig(fig_path, dpi=150)
                plt.close()
                print(f"✅ Saved: {fig_path}")

            except Exception as e:
                print(f"⚠️ Error in {layer}-{model_name}: {e}")




✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\plots_nilearn_top5+5\connectome_top5_control_ms_FA_GAT.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\plots_nilearn_top5+5\connectome_top5_control_ms_GM_GAT.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\plots_nilearn_top5+5\connectome_top5_control_ms_rsfMRI_GAT.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_layer/figures/ML/gnn_explainer\plots_nilearn_top5+5\connectome_top5_control_ms_FA_GCN.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATO